In [0]:

# Insert into Gold orders Table using Spark SQL

def validations(start_date,end_date):
    try:
        # Check if the sivler table has data for the given date range.
        if len(spark.table("brazilian_ecommerce.silver.orders").filter(f"order_purchase_timestamp between '{start_date}' and '{end_date}'").take(1)) == 0:
            raise Exception("Data is missing the silver layer orders table")
        elif len(spark.table("brazilian_ecommerce.silver.customers").take(1)) ==0:
            raise Exception("Data is missing the silver layer customers table")
        elif len(spark.table("brazilian_ecommerce.silver.sellers").take(1)) ==0:
            raise Exception("Data is missing the silver layer sellers table")
        elif len(spark.table("brazilian_ecommerce.silver.products").take(1)) ==0:
            raise Exception("Data is missing the silver layer products table")
        elif len(spark.table("brazilian_ecommerce.gold.orders").filter(f"order_date between '{start_date}' and '{end_date}'").take(1)) >0:
            raise Exception("Data is already present in the gold layer orders table")
        else:
            print("All validations passed")
    except Exception as e:
        raise Exception(f"Error: {e}")

def gold_ingestion(start_date,end_date):
    try:
        
        # Perform basic validations on the silvere table
        validations(start_date,end_date)

        spark.sql(f"""INSERT INTO brazilian_ecommerce.gold.orders
        WITH orders_prepared AS (
            SELECT
                customer_state,
                customer_city,
                seller_state,
                seller_city,
                order_status,
                c.product_category_name_english AS product_category_english,
                CASE 
                    WHEN lower(customer_state) =lower(seller_state) AND lower(customer_city) = lower(seller_city) THEN 'Local Delivery'
                    WHEN lower(customer_state) = lower(seller_state) THEN 'Intra-State Delivery'
                    ELSE 'Inter-State Delivery'
                END AS delivery_type,
                CASE 
                    WHEN payment_installments = 1 THEN 'Single Payment'
                    ELSE 'EMI'
                END AS cleansed_payment_type,
                YEAR(order_purchase_timestamp) AS order_year,
                MONTH(order_purchase_timestamp) AS order_month,
                DATE(order_purchase_timestamp) AS order_date,
                DATEDIFF(order_delivered_customer_date, order_approved_at) AS delivery_days,
                DATEDIFF(order_delivered_customer_date, order_delivered_carrier_date) AS shipping_time_days,
                CASE WHEN DATE(order_estimated_delivery_date) = DATE(order_delivered_customer_date) THEN 1 ELSE 0 END AS on_time_delivery,
                price,
                freight_value,
                price + freight_value AS item_total_value,
                b.customer_id,
                d.seller_id,
                c.product_id
            FROM brazilian_ecommerce.silver.orders a
            INNER JOIN brazilian_ecommerce.silver.customers b ON a.customer_id = b.customer_id and (cast(order_purchase_timestamp as date) between b.valid_from and b.valid_to)
            INNER JOIN brazilian_ecommerce.silver.products c ON a.product_id = c.product_id and (cast(order_purchase_timestamp as date) between c.valid_from and b.valid_to)
            INNER JOIN brazilian_ecommerce.silver.sellers d ON a.seller_id = d.seller_id and (cast(order_purchase_timestamp as date) between c.valid_from and b.valid_to)
            WHERE a.order_id IS NOT NULL and a.order_purchase_timestamp between '{start_date}' and '{end_date}'
        )
        SELECT
            customer_state,
            customer_city,
            seller_state,
            seller_city,
            order_status,
            delivery_type,
            cleansed_payment_type,
            order_year,
            order_month,
            order_date,
            on_time_delivery,
            product_category_english,
            MAX(delivery_days) AS deliver_days,
            MAX(shipping_time_days) AS shipping_time_days,
            SUM(price) AS price,
            SUM(freight_value) AS freight_value,
            SUM(item_total_value) AS item_total_value,
            COUNT(DISTINCT customer_id) AS total_customers,
            COUNT(DISTINCT seller_id) AS total_sellers,
            COUNT(DISTINCT product_id) AS total_products
        FROM orders_prepared
        GROUP BY
            customer_state,
            customer_city,
            seller_state,
            seller_city,
            order_status,
            delivery_type,
            cleansed_payment_type,
            order_year,
            order_month,
            order_date,
            on_time_delivery,
            product_category_english
        """)
        print("Data was successfully inserted to the gold layer table.")

    except Exception as e:
        raise Exception(f"Error: {e}")


if __name__ == "__main__":
    print("Gold Ingestion Started")
    start_date='2016-01-01'
    end_date='2018-12-31'
    gold_ingestion(start_date,end_date)
    print("Gold Ingestion Completed")


Gold Ingestion Started


---------------------------------------------------------------------------
Exception                                 Traceback (most recent call last)
File <command-6912169315332272>, line 16, in validations(start_date, end_date)
     15 elif len(spark.table("brazilian_ecommerce.gold.orders").filter(f"order_date between '{start_date}' and '{end_date}'").take(1)) >0:
---> 16     raise Exception("Data is already present in the gold layer orders table")
     17 else:

Exception: Data is already present in the gold layer orders table

During handling of the above exception, another exception occurred:

Exception                                 Traceback (most recent call last)
File <command-6912169315332272>, line 26, in gold_ingestion(start_date, end_date)
     23 try:
     24     
     25     # Perform basic validations on the silvere table
---> 26     validations(start_date,end_date)
     28     spark.sql(f"""INSERT INTO brazilian_ecommerce.gold.orders
     29     WITH orders_prepared 